# Didactic Quality Check — версия с жюри моделей

Грузим **результат генерации** (README проекта) и проверяем его по **дидактической оси**.
В этой версии судит не одна модель, а **жюри (panel) из нескольких разных моделей**: каждая
ставит балл независимо, мы берём **медиану**, а разброс между моделями превращаем в уверенность.
Спорные дименшены уходят в **разномодельную дискуссию** (критик / защитник / судья — разные модели).

Два режима запуска:
- **LLM-режим** — если есть `OPENROUTER_API_KEY`, жюри = реальные модели (gemini / deepseek / qwen / …).
- **Heuristic-режим (mock)** — без ключа жюри имитируется: каждая «модель» смотрит на документ
  через свою линзу и ставит чуть разные баллы. Это позволяет увидеть несогласие панели и эскалацию
  без единого API-вызова. Mock реально считает повторы, сломанные таблицы и рассинхрон диаграмм.


## Принцип (коротко)

```
Ось 1 — Структура (39 критериев, существующий RubricScorer). Здесь только лёгкий pre-scan.
Ось 2 — Дидактика (этот notebook): 1-5 по дименшенам, судит жюри моделей.
```
Оси не складываются: `39/39` структуры при `2.6/5` дидактики — валидный результат.

Внутри оси 2:
1. **Жюри (PoLL).** N разных моделей оценивают дименшен независимо → медиана.
2. **Разброс = уверенность.** Сошлись — верим; разбежались — это спорный случай.
3. **Эскалация в дискуссию** для спорных / проваленных дименшенов: критик/защитник/судья на разных моделях.
4. **Отказ вместо ложного балла:** низкая уверенность или провал порога → на человека.


## Почему жюри, а не один судья (и почему судья ≠ генератор)

Две подтверждённые проблемы одиночного судьи:
- **Предвзятость к себе.** Модель выше оценивает тексты, похожие на её собственные. Если контент
  генерит GPT, нельзя, чтобы его же и судил GPT — поэтому генератор **исключается** из жюри.
- **Общие слепые пятна.** Четыре прогона одной модели — это один и тот же мозг четыре раза:
  уверенное единогласие, которое ничего не доказывает.

Жюри из разных семейств моделей гасит несвязанные ошибки и, по данным Cohere (PoLL, arXiv 2404.18796),
несколько меньших моделей обходят одну большую — точнее, менее предвзято и в разы дешевле.
Важно: **единого «лучшего» судьи нет** — состав жюри подбирается под задачу по согласию с экспертами,
а не угадывается (см. финальную секцию).

Агрегируем **медианой**, а не средним: медиана устойчива к одному «дикому» мнению, а голосование
теряет информацию на шкале 1-5.


## Настройки запуска

In [1]:
import os, re, json, statistics, urllib.request
from pathlib import Path
from dataclasses import dataclass, field
from collections import Counter

# --- LLM (опционально). Ключ ТОЛЬКО из окружения. ---
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "").strip()
USE_LLM = bool(OPENROUTER_API_KEY)

# Модель-ГЕНЕРАТОР контента. Она ИСКЛЮЧАЕТСЯ из жюри (анти-self-bias).
GENERATOR_MODEL = os.environ.get("GENERATOR_MODEL", "openai/gpt-5.4-mini")

# Жюри: разные семейства. В mock-режиме имена условные, важны сами различия линз.
JURY_MODELS = [
    "google/gemini-3.1-flash-lite",
    "deepseek/deepseek-v4-pro",
    "qwen/qwen3.7-max",
]
# Пул для дискуссии (роли раздаём РАЗНЫМ моделям).
DEBATE_ROLES = {
    "critic":   "deepseek/deepseek-v4-pro",
    "defender": "qwen/qwen3.7-max",
    "judge":    "google/gemini-3.1-flash-lite",
}

# --- Параметры дидактической оси ---
ABSTAIN_CONFIDENCE = 0.55   # ниже -> эскалация / на человека
DIDACTIC_FLOOR     = 3.0    # дименшен ниже -> major issue + эскалация
DEBATE_ON_ESCALATE = True
DEBATE_ROUNDS      = 1

# анти-self-bias: выкидываем генератор из жюри и из ролей
removed = [m for m in JURY_MODELS if m == GENERATOR_MODEL]
JURY_MODELS = [m for m in JURY_MODELS if m != GENERATOR_MODEL]
if removed:
    print(f"⚠ Из жюри исключён генератор {GENERATOR_MODEL} (анти-self-bias).")
assert len(JURY_MODELS) >= 2, "Нужно >=2 моделей в жюри после исключения генератора."

def find_doc():
    for base in ["/mnt/user-data/uploads", ".", "/mnt/data"]:
        p = Path(base)
        if p.exists():
            hits = sorted(p.glob("*.md"))
            if hits: return hits[0]
    return None
DOC_PATH = Path(os.environ["DOC_PATH"]) if os.environ.get("DOC_PATH") else find_doc()

print("Режим      :", "LLM" if USE_LLM else "HEURISTIC MOCK (нет OPENROUTER_API_KEY)")
print("Генератор  :", GENERATOR_MODEL, "(исключён из жюри)")
print("Жюри       :", JURY_MODELS)
print("Дискуссия  :", {k: v for k, v in DEBATE_ROLES.items()})
print("Документ   :", DOC_PATH)

Режим      : HEURISTIC MOCK (нет OPENROUTER_API_KEY)
Генератор  : openai/gpt-5.4-mini (исключён из жюри)
Жюри       : ['google/gemini-3.1-flash-lite', 'deepseek/deepseek-v4-pro', 'qwen/qwen3.7-max']
Дискуссия  : {'critic': 'deepseek/deepseek-v4-pro', 'defender': 'qwen/qwen3.7-max', 'judge': 'google/gemini-3.1-flash-lite'}
Документ   : /mnt/user-data/uploads/PjM15_PublicSpeaking.md


In [2]:
assert DOC_PATH and DOC_PATH.exists(), "Не найден .md документ."
MD = DOC_PATH.read_text(encoding="utf-8")
print(f"Загружено: {DOC_PATH.name}  ({len(MD)} символов, {MD.count(chr(10))+1} строк)")

Загружено: PjM15_PublicSpeaking.md  (22594 символов, 269 строк)


## Лёгкий структурный pre-scan (ось 1, для контраста)

In [3]:
def structural_prescan(md: str) -> dict:
    return {
        "has_h1": bool(re.search(r"^#\s+\S", md, re.M)),
        "has_toc": bool(re.search(r"^##\s+(Содержание|Оглавление)", md, re.M)),
        "chapter1": bool(re.search(r"^##\s+Глава\s*1", md, re.M)),
        "chapter2": bool(re.search(r"^##\s+Глава\s*2", md, re.M)),
        "chapter3": bool(re.search(r"^##\s+Глава\s*3", md, re.M)),
        "theory_parts": len(re.findall(r"^###\s+2\.\d+\.", md, re.M)),
        "tasks": len(re.findall(r"^###\s+Задани", md, re.M)),
        "examples": len(re.findall(r"\*\*Пример", md)),
    }
pre = structural_prescan(MD)
ok = sum(1 for k in ["has_h1","has_toc","chapter1","chapter2","chapter3"] if pre[k])
for k, v in pre.items(): print(f"  {k:14}: {v}")
print(f"\n=> Каркас {ok}/5 на месте. Формально документ выглядит готовым.")

  has_h1        : True
  has_toc       : True
  chapter1      : True
  chapter2      : True
  chapter3      : True
  theory_parts  : 3
  tasks         : 3
  examples      : 3

=> Каркас 5/5 на месте. Формально документ выглядит готовым.


## Дидактические дименшены (стартовый набор-гипотеза)

In [4]:
DIMENSIONS = {
    "coherence":       {"title": "Связность",
        "question": "Единый маршрут без разрывов, оборванных фраз и скачков?"},
    "scaffolding":     {"title": "Scaffolding (теория готовит к практике)",
        "question": "Теория главы 2 реально готовит к заданиям главы 3?"},
    "example_quality": {"title": "Качество примеров",
        "question": "Примеры конкретны и раскрывают идею, а не заглушки?"},
    "cognitive_load":  {"title": "Когнитивная нагрузка",
        "question": "Нет повторов и перегруза, адекватная прогрессия?"},
    "school_tone":     {"title": "Тон школы (p2p)",
        "question": "Peer-тон: не директивно, решение не выдаётся?"},
    "naturalness":     {"title": "Не-AI-водность",
        "question": "Живой язык без шаблонных самоповторов?"},
}
print("Дименшенов:", len(DIMENSIONS))

Дименшенов: 6


## Эвристические сигналы (evidence + основа mock-моделей)

In [5]:
def _sentences(t):
    t = re.sub(r"```.*?```", " ", t, flags=re.S)
    t = re.sub(r"<[^>]+>", " ", t)
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", t) if len(s.strip()) > 25]

def repetition_ratio(t, n=8):
    w = re.findall(r"\w+", t.lower())
    grams = [" ".join(w[i:i+n]) for i in range(len(w)-n+1)]
    if not grams: return 0.0
    c = Counter(grams)
    return sum(v for v in c.values() if v > 1) / len(grams)

def near_duplicate_pairs(t, thr=0.7, cap=40):
    sents = _sentences(t); sets = [set(re.findall(r"\w+", s.lower())) for s in sents]; pairs = []
    for i in range(len(sets)):
        for j in range(i+1, len(sets)):
            a, b = sets[i], sets[j]
            if a and b and len(a & b) / len(a | b) >= thr:
                pairs.append((sents[i][:80], sents[j][:80]))
                if len(pairs) >= cap: return pairs
    return pairs

def broken_table_rows(t):
    return [ln[:100] for ln in t.splitlines() if ln.count("|") >= 2 and len(ln) > 200]

def _nearest_heading(t, idx):
    head = None
    for m in re.finditer(r"^#{2,3}\s+(.+)$", t[:idx], flags=re.M): head = m.group(1)
    return head

def diagram_topic_match(t):
    out = []
    for m in re.finditer(r"```mermaid(.*?)```", t, flags=re.S):
        head = _nearest_heading(t, m.start())
        nodes = set(re.findall(r"[А-Яа-яЁё]{4,}", m.group(1).lower()))
        hw = set(re.findall(r"[А-Яа-яЁё]{4,}", (head or "").lower()))
        jac = len(nodes & hw) / len(nodes | hw) if (nodes | hw) else 0.0
        out.append(round(jac, 3))
    return out

def collect_signals(md: str) -> dict:
    dm = diagram_topic_match(md)
    return {
        "repetition_ratio": round(repetition_ratio(md), 3),
        "near_dup": len(near_duplicate_pairs(md)),
        "near_dup_examples": near_duplicate_pairs(md)[:2],
        "broken_tables": len(broken_table_rows(md)),
        "diagram_match_avg": round(sum(dm) / len(dm), 3) if dm else 1.0,
        "example_count": len(re.findall(r"\*\*Пример", md)),
        "directive_hits": len(re.findall(r"\b(сделай|нажми|введите|скопируй|выполни шаг)\b", md.lower())),
    }

SIGNALS = collect_signals(MD)
for k, v in SIGNALS.items():
    if k != "near_dup_examples": print(f"  {k:18}: {v}")

  repetition_ratio  : 0.147
  near_dup          : 17
  broken_tables     : 1
  diagram_match_avg : 0.035
  example_count     : 3
  directive_hits    : 1


## Модель отчёта (вторая ось; per-model баллы внутри дименшена)

In [6]:
@dataclass
class DidacticScore:
    dimension: str
    score: float                          # медиана по жюри (после возможной дискуссии)
    confidence: float                     # из разброса между моделями
    per_model: dict = field(default_factory=dict)   # {model: balls}
    rationale: str = ""
    evidence: list = field(default_factory=list)
    escalated: bool = False
    escalate_reason: str = ""
    debate_transcript: list = field(default_factory=list)

@dataclass
class DidacticQualityReport:
    dimensions: list
    overall_raw: float = 0.0
    overall_calibrated: float = None
    calibrated: bool = False
    needs_human_review: bool = False
    abstain_reasons: list = field(default_factory=list)
    jury: list = field(default_factory=list)
    n_jury: int = 0

## LLM-клиент (OpenRouter, plain JSON). Без ключа не вызывается.

In [7]:
def call_llm_json(model: str, system_prompt: str, user_prompt: str, max_tokens=900) -> dict:
    payload = {"model": model, "max_tokens": max_tokens, "messages": [
        {"role": "system", "content": system_prompt + "\nReturn ONLY valid JSON, no fences."},
        {"role": "user", "content": user_prompt}]}
    req = urllib.request.Request("https://openrouter.ai/api/v1/chat/completions",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}", "Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=120) as r:
        data = json.loads(r.read().decode("utf-8"))
    c = data["choices"][0]["message"]["content"].strip()
    c = re.sub(r"^```(?:json)?|```$", "", c).strip()
    s, e = c.find("{"), c.rfind("}")
    return json.loads(c[s:e+1])

## Один член жюри оценивает один дименшен

LLM-путь — реальный промпт на конкретной модели. Mock-путь — каждая «модель» смотрит через
свою линзу (свои строгости по дименшенам), поэтому баллы расходятся — как у настоящих разных моделей.


In [8]:
def _base_mock_score(dim_id, signals):
    """Базовый эвристический балл 1-5 + объяснение из реальных сигналов."""
    rep, ndup = signals["repetition_ratio"], signals["near_dup"]
    broken, match = signals["broken_tables"], signals["diagram_match_avg"]
    if dim_id == "naturalness":
        return 5 - min(3.2, rep*18 + ndup*0.06), \
            "Самоповторы: одни и те же связки-болванки дословно.", \
            [f"{ndup} почти-дублей предложений", f"repetition_ratio={rep}"]
    if dim_id == "coherence":
        return 5 - min(2.8, broken*1.0 + ndup*0.04 + (1 if match < 0.2 else 0)), \
            "Разрывы потока: сломанная таблица и диаграммы не по теме.", \
            [f"{broken} разваленных таблиц", f"диаграммы вне темы (match={match})"]
    if dim_id == "cognitive_load":
        return 5 - min(2.7, rep*15 + ndup*0.05), \
            "Повторы раздувают объём без смысла — лишняя нагрузка.", \
            [f"{ndup} дубль-предложений"]
    if dim_id == "example_quality":
        return (3.7 if signals["example_count"] >= 3 else 2.5), \
            "Примеры есть, но слабо привязаны к кейсу проекта.", \
            [f"примеров: {signals['example_count']} (обобщённые)"]
    if dim_id == "scaffolding":
        return 3.1, "Теория местами generic, опора под практику размыта.", \
            ["диаграммы-болванки не отражают теорию"]
    if dim_id == "school_tone":
        return (4.2 if signals["directive_hits"] == 0 else 3.0), \
            "Тон p2p выдержан: правила, не готовые ответы.", \
            [f"директив: {signals['directive_hits']}"]
    return 3.0, "—", []

# Линзы mock-моделей: насколько каждая строга по дименшену (смещение к базовому баллу).
# Намеренно разные -> жюри расходится, как настоящие разные модели.
MOCK_LENS = {
    "google/gemini-3.1-flash-lite": {"naturalness":-0.5,"coherence":-0.3,"scaffolding":-0.9,
        "example_quality":-1.5,"cognitive_load":-0.3,"school_tone":0.0},
    "deepseek/deepseek-v4-pro":     {"naturalness":-0.2,"coherence":-0.5,"scaffolding":+0.1,
        "example_quality":-0.3,"cognitive_load":-0.1,"school_tone":-0.2},
    "qwen/qwen3.7-max":             {"naturalness":+0.3,"coherence":+0.2,"scaffolding":+0.4,
        "example_quality":+1.1,"cognitive_load":+0.3,"school_tone":+0.3},
}

def juror_score(model, dim_id, spec, signals, learning_outcomes):
    if USE_LLM:
        sys = ("Ты — методист школы проектного p2p-обучения. Оцени README по ОДНОМУ дидактическому "
               "критерию. Сначала рассуждение, потом балл 1-5 (5=отлично). Формальное наличие блока "
               "не равно качеству.")
        usr = (f"КРИТЕРИЙ: {spec['title']}\nВОПРОС: {spec['question']}\n"
               f"ЗУНы: {learning_outcomes or '—'}\n\nREADME (между <<< >>>):\n<<<\n{MD[:11000]}\n>>>\n\n"
               "JSON: {\"rationale\": str, \"evidence\": [цитаты], \"score\": 1-5}")
        try:
            o = call_llm_json(model, sys, usr)
            return float(o["score"]), o.get("rationale", ""), o.get("evidence", [])[:3]
        except Exception as e:
            print(f"  [warn] {model} упал на {dim_id}: {e}; mock")
    base, rat, ev = _base_mock_score(dim_id, signals)
    bias = MOCK_LENS.get(model, {}).get(dim_id, 0.0)
    return round(max(1.0, min(5.0, base + bias)), 2), rat, ev

## Жюри: панель оценивает дименшен → медиана + уверенность из разброса

In [9]:
def jury_score_dimension(dim_id, spec, signals, learning_outcomes):
    per_model, rats, evs = {}, [], []
    for m in JURY_MODELS:
        s, rat, ev = juror_score(m, dim_id, spec, signals, learning_outcomes)
        per_model[m] = s; rats.append(rat); evs += ev
    scores = list(per_model.values())
    median = round(statistics.median(scores), 2)
    spread = statistics.pstdev(scores) if len(scores) > 1 else 0.0
    confidence = round(max(0.0, 1.0 - spread / 2.0), 2)
    # сводный rationale + уникальные evidence
    seen, uniq_ev = set(), []
    for e in evs:
        if e not in seen: seen.add(e); uniq_ev.append(e)
    return DidacticScore(dimension=dim_id, score=median, confidence=confidence,
                         per_model=per_model, rationale=rats[0], evidence=uniq_ev[:4])

## Дискуссия на спорном дименшене (роли — разные модели)

Запускается как **эскалация**: дименшен ниже пола или жюри не сошлось.
critic / defender / judge — три РАЗНЫЕ модели (критик и судья не совпадают, иначе спор бессмыслен).


In [10]:
def debate_dimension(dim_id, spec, signals, jury_scores, learning_outcomes):
    transcript = []
    if USE_LLM:
        base = (f"Критерий: {spec['title']}. Вопрос: {spec['question']}. "
                f"README:\n{MD[:8000]}")
        try:
            crit = call_llm_json(DEBATE_ROLES["critic"],
                "Ты CRITIC: жёстко найди дидактические слабости по критерию. JSON {\"points\":[...]}", base)
            transcript.append(("critic", DEBATE_ROLES["critic"], crit.get("points", [])))
            deff = call_llm_json(DEBATE_ROLES["defender"],
                "Ты DEFENDER: ответь критику, отметь сильное. JSON {\"points\":[...]}",
                base + "\nКритика: " + json.dumps(crit, ensure_ascii=False))
            transcript.append(("defender", DEBATE_ROLES["defender"], deff.get("points", [])))
            verd = call_llm_json(DEBATE_ROLES["judge"],
                "Ты JUDGE: взвесь обе стороны, балл 1-5. JSON {\"score\":num,\"rationale\":str}",
                base + "\nCRITIC:" + json.dumps(crit, ensure_ascii=False) +
                "\nDEFENDER:" + json.dumps(deff, ensure_ascii=False))
            transcript.append(("judge", DEBATE_ROLES["judge"], verd.get("rationale", "")))
            return float(verd["score"]), verd.get("rationale", ""), transcript
        except Exception as e:
            print(f"  [warn] LLM-дискуссия упала на {dim_id}: {e}; mock")
    # mock: критик = строгая модель, защитник = мягкая, судья = третья (взвешивает)
    strict = min(jury_scores, key=jury_scores.get)
    lenient = max(jury_scores, key=jury_scores.get)
    # судья = модель, отличная и от критика, и от защитника (критик != судья)
    rest = [m for m in JURY_MODELS if m not in (strict, lenient)]
    judge_model = rest[0] if rest else DEBATE_ROLES["judge"]
    _, crit_rat, crit_ev = _base_mock_score(dim_id, signals)
    transcript.append(("critic", strict, crit_ev or [crit_rat]))
    transcript.append(("defender", lenient, ["формальные требования соблюдены; блок заполнен"]))
    # судья ближе к критику, но учитывает защиту -> чуть выше минимума
    final = round((jury_scores[strict] * 0.6 + jury_scores[lenient] * 0.4), 2)
    rat = f"После спора критика (повторы/разрывы) перевешивает формальную полноту. {crit_rat}"
    transcript.append(("judge", judge_model, rat))
    return final, rat, transcript

## Сборка: жюри + эскалация в дискуссию

In [11]:
def judge_dimension(dim_id, spec, signals, learning_outcomes):
    sc = jury_score_dimension(dim_id, spec, signals, learning_outcomes)
    low_conf = sc.confidence < ABSTAIN_CONFIDENCE
    below = sc.score < DIDACTIC_FLOOR
    if DEBATE_ON_ESCALATE and (low_conf or below):
        sc.escalated = True
        sc.escalate_reason = ("разброс жюри" if low_conf else "") + \
                             (" + ниже пола" if (low_conf and below) else ("ниже пола" if below else ""))
        d_score, d_rat, d_trans = debate_dimension(dim_id, spec, signals, sc.per_model, learning_outcomes)
        sc.score, sc.rationale, sc.debate_transcript = d_score, d_rat, d_trans
    return sc

## Прогон по всем дименшенам

In [12]:
LEARNING_OUTCOMES = ["Справляться с волнением перед выступлением",
                     "Готовить структурированную речь", "Уверенно отвечать на вопросы"]
dims = []
for dim_id, spec in DIMENSIONS.items():
    print(f"… жюри оценивает: {spec['title']}")
    dims.append(judge_dimension(dim_id, spec, SIGNALS, LEARNING_OUTCOMES))

overall = round(statistics.median([d.score for d in dims]), 2)
abstain = []
for d in dims:
    if d.confidence < ABSTAIN_CONFIDENCE: abstain.append(f"jury_split:{d.dimension}")
    if d.score < DIDACTIC_FLOOR: abstain.append(f"below_floor:{d.dimension}")
report = DidacticQualityReport(dimensions=dims, overall_raw=overall,
    needs_human_review=bool(abstain), abstain_reasons=sorted(set(abstain)),
    jury=JURY_MODELS, n_jury=len(JURY_MODELS))
print("\nГотово.")

… жюри оценивает: Связность
… жюри оценивает: Scaffolding (теория готовит к практике)
… жюри оценивает: Качество примеров
… жюри оценивает: Когнитивная нагрузка
… жюри оценивает: Тон школы (p2p)
… жюри оценивает: Не-AI-водность

Готово.


## Итог: две оси рядом + раскладка по жюри

In [13]:
short = lambda m: m.split("/")[-1][:14]
print("="*70)
print(f"ДОКУМЕНТ: {DOC_PATH.name}")
print("="*70)
print(f"\nОСЬ 1 (структура): каркас {ok}/5 — формально готов.")
print(f"\nОСЬ 2 (дидактика): overall = {report.overall_raw}/5  [НЕ калибровано под эксперта]")
print(f"Жюри ({report.n_jury}): {', '.join(short(m) for m in report.jury)}\n")

cols = "".join(f"{short(m):>16}" for m in JURY_MODELS)
print(f"{'дименшен':24}{cols}{'медиана':>9}{'conf':>7}  эскалация")
print("-"*(24+16*len(JURY_MODELS)+16+12))
for d in dims:
    row = "".join(f"{d.per_model[m]:>16}" for m in JURY_MODELS)
    esc = ("MAD: " + d.escalate_reason) if d.escalated else ""
    print(f"{DIMENSIONS[d.dimension]['title'][:24]:24}{row}{d.score:>9}{d.confidence:>7}  {esc}")

print("\nВЕРДИКТ:")
if report.needs_human_review:
    print("  ⚠ needs_human_review = True ->", ", ".join(report.abstain_reasons))
    print("  (в проде поднимает human_review_required в MethodologyGate, стадия evaluation)")
else:
    print("  ✓ блокеров нет")

print("\nПояснения:")
for d in dims:
    tag = " [после дискуссии]" if d.escalated else ""
    print(f"\n• {DIMENSIONS[d.dimension]['title']} — {d.score}/5 (conf {d.confidence}){tag}")
    print(f"    {d.rationale}")
    for e in d.evidence: print(f"    - {e}")

ДОКУМЕНТ: PjM15_PublicSpeaking.md

ОСЬ 1 (структура): каркас 5/5 — формально готов.

ОСЬ 2 (дидактика): overall = 2.62/5  [НЕ калибровано под эксперта]
Жюри (3): gemini-3.1-fla, deepseek-v4-pr, qwen3.7-max

дименшен                  gemini-3.1-fla  deepseek-v4-pr     qwen3.7-max  медиана   conf  эскалация
----------------------------------------------------------------------------------------------------
Связность                           2.02            1.82            2.52      2.1   0.85  MAD: ниже пола
Scaffolding (теория гото             2.2             3.2             3.5      3.2   0.72  
Качество примеров                    2.2             3.4             4.8     3.24   0.47  MAD: разброс жюри
Когнитивная нагрузка                 2.0             2.2             2.6     2.24   0.88  MAD: ниже пола
Тон школы (p2p)                      3.0             2.8             3.3      3.0    0.9  
Не-AI-водность                       1.3             1.6             2.1     1.62   0.84  MA

## Транскрипты дискуссий

In [14]:
debated = [d for d in dims if d.escalated]
if not debated: print("Эскалаций не было.")
for d in debated:
    print("="*64); print(f"ДИСКУССИЯ: {DIMENSIONS[d.dimension]['title']} ({d.escalate_reason}) -> {d.score}/5"); print("="*64)
    for role, model, content in d.debate_transcript:
        print(f"\n[{role.upper()}]  ({model.split('/')[-1]})")
        if isinstance(content, list):
            for c in content: print("  •", c)
        else: print(" ", content)

ДИСКУССИЯ: Связность (ниже пола) -> 2.1/5

[CRITIC]  (deepseek-v4-pro)
  • 1 разваленных таблиц
  • диаграммы вне темы (match=0.035)

[DEFENDER]  (qwen3.7-max)
  • формальные требования соблюдены; блок заполнен

[JUDGE]  (gemini-3.1-flash-lite)
  После спора критика (повторы/разрывы) перевешивает формальную полноту. Разрывы потока: сломанная таблица и диаграммы не по теме.
ДИСКУССИЯ: Качество примеров (разброс жюри) -> 3.24/5

[CRITIC]  (gemini-3.1-flash-lite)
  • примеров: 3 (обобщённые)

[DEFENDER]  (qwen3.7-max)
  • формальные требования соблюдены; блок заполнен

[JUDGE]  (deepseek-v4-pro)
  После спора критика (повторы/разрывы) перевешивает формальную полноту. Примеры есть, но слабо привязаны к кейсу проекта.
ДИСКУССИЯ: Когнитивная нагрузка (ниже пола) -> 2.24/5

[CRITIC]  (gemini-3.1-flash-lite)
  • 17 дубль-предложений

[DEFENDER]  (qwen3.7-max)
  • формальные требования соблюдены; блок заполнен

[JUDGE]  (deepseek-v4-pro)
  После спора критика (повторы/разрывы) перевешивает форм

## Как читать и что дальше

**Что показал прогон.** Структурно документ готов, а дидактика проседает. Разные модели жюри
ставят РАЗНЫЕ баллы (видно в колонках): где они сходятся низко — уверенный провал; где расходятся —
дименшен спорный и уходит в дискуссию critic/defender/judge на разных моделях.

**Что честно, а что заглушка.**
- Механика реальна: жюри, медиана, уверенность из разброса, эскалация, дискуссия с разными ролями,
  исключение генератора из жюри.
- Без ключа модели имитируются линзами `MOCK_LENS` — это демонстрирует НЕСОГЛАСИЕ, но не настоящие
  модели. С `OPENROUTER_API_KEY` жюри = реальные gemini/deepseek/qwen.

**Чтобы стало доказуемым (по приоритету):**
1. **Экспертный набор 30-50 README** (с «39/39, но забраковано»). Без него `overall_calibrated`=None.
2. **Подбор состава жюри по данным.** Единого лучшего судьи нет — измерь согласие КАЖДОЙ модели
   с экспертом по каждому дименшену (QWK/Spearman/ICC) и выкини плохие. Состав не угадывается.
3. Парное сравнение с банком эталонов (нет на входе) + калибровка raw→экспертный балл.

**Запуск с реальными моделями:**
`OPENROUTER_API_KEY=...  GENERATOR_MODEL=...  DOC_PATH=/путь/README.md` и перезапусти.
Поменяй состав `JURY_MODELS` / `DEBATE_ROLES` под свои доступные слаги OpenRouter.
